In [1]:
import socket
from typing import Optional

from langchain_core.tools import tool

from src.apps.fuse import meta


def get_local_ip(family: int = socket.AF_INET) -> str:
    """获取本机对外 IP（兼容 IPv4 / IPv6 / 容器）。

    Args:
        family: socket.AF_INET (IPv4) 或 socket.AF_INET6 (IPv6)

    Returns:
        IP 字符串；失败返回 127.0.0.1
    """
    try:
        s = socket.socket(family, socket.SOCK_DGRAM)
        try:
            target = ("8.8.8.8", 80) if family == socket.AF_INET else ("2001:4860:4860::8888", 80)
            s.connect(target)
            return s.getsockname()[0]
        finally:
            s.close()
    except Exception:
        # 回退到 hostname 解析
        try:
            return socket.gethostbyname(socket.gethostname())
        except Exception:
            return "127.0.0.1"


# IPv4
ipv4 = get_local_ip(socket.AF_INET)
print(f"IPv4: {ipv4}")

IPv4: 28.0.0.1


In [2]:
import os
import uuid
from unittest import result

import dotenv
from langchain_openai import ChatOpenAI
from langfuse import get_client as get_langfuse_client
from langfuse.langchain import CallbackHandler

from langchain.agents import create_agent
dotenv.load_dotenv(override=True)


True

In [3]:
from src.apps.fuse.meta import IdentityConfig

# Initialize Langfuse client
langfuse = get_langfuse_client()

# Initialize Langfuse CallbackHandler for Langchain (tracing)
langfuse_handler = CallbackHandler()


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together and return the result."""
    return a + b

model = ChatOpenAI(
    model_name="MiniMax-M2.7",
    api_key=os.getenv("MINIMAX_API_KEY"),
    base_url=os.getenv("MINIMAX_BASE_URL"),
)
agent = create_agent(
    model=model,
    tools=[add_numbers],
    system_prompt="You are a helpful math tutor who can do calculations using the provided tools.",
)

conversation_id= uuid.uuid4().hex[:8]
session_id = f"sess-{conversation_id}"  # 从前端 cookie / URL 取

import hashlib
ipv4 = get_local_ip(socket.AF_INET)

user_id = f"{ipv4}-{conversation_id}"
trace_name = f"ReAct-agent-v2-{conversation_id}"        # 按 batch
tags = [
  os.getenv("ENV", "dev"),          # prod / staging / dev
  "agent",                          # 业务线
  "v0.0.1",                         # agent 版本（低基数）
]

identity = IdentityConfig(
    session_id=session_id,
    user_id=user_id,
    trace_name=trace_name,
    tags=tags
)

agent_meta = meta.AgentMeta(
    agent_name="ReAct-agent-v2",
)
trace_metadata = meta.TraceMetadata(
    identity=identity,
    agent=agent_meta,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is 42 + 58?"}]},
    # config={"callbacks": [langfuse_handler]}
    config=trace_metadata.to_langchain_config(callbacks=[langfuse_handler])
)
from rich import print as rprint
# rprint(result)

In [4]:
rprint(result["messages"][-1].content)


<think>The calculator has returned 100 as the result. I can now provide the final answer to the user.
</think>


42 + 58 = **100**